In [2]:
import json
import re
from bs4 import BeautifulSoup

def clean_wiki_text(text):
    """Làm sạch văn bản, loại bỏ các tham chiếu [1], [2] và ký tự rác."""
    if not text: return ""
    # Xóa tham chiếu chú thích dạng [1], [2], [ghi chú 1]
    text = re.sub(r'\[[^\]]*\]', '', text)
    # Chuẩn hóa khoảng trắng và ngắt dòng
    text = text.replace('\xa0', ' ').replace('\n', ' ')
    return re.sub(r'\s+', ' ', text).strip()

def extract_comprehensive_data(file_path, output_json):
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')

    # Tìm vùng nội dung chính của Wikipedia
    content_body = soup.find('div', class_='mw-parser-output')
    if not content_body:
        print("Không tìm thấy vùng nội dung chính.")
        return

    result_data = {
        "metadata": {
            "title": soup.find('h1', id='firstHeading').get_text(strip=True) if soup.find('h1', id='firstHeading') else "N/A"
        },
        "sections": {},
        "tables": []
    }

    # 1. Trích xuất tất cả văn bản theo Section (h2, h3)
    current_section = "Giới thiệu"
    for elem in content_body.find_all(['h2', 'h3', 'p', 'ul', 'ol']):
        if elem.name in ['h2', 'h3']:
            # Lấy tên tiêu đề mục
            header_text = clean_wiki_text(elem.get_text().replace('sửa', '').replace('mã nguồn', ''))
            if header_text in ["Ghi chú", "Tham khảo", "Xem thêm", "Liên kết ngoài", "Thư mục"]:
                current_section = None
            else:
                current_section = header_text
        elif current_section:
            text = clean_wiki_text(elem.get_text())
            if text:
                result_data["sections"].setdefault(current_section, []).append(text)

    # 2. Trích xuất Bảng Danh sách Thủ khoa (Xử lý chi tiết thẻ td, th)
    tables = content_body.find_all('table', class_='wikitable')
    for table_idx, table in enumerate(tables):
        table_info = {"table_id": table_idx + 1, "data": []}
        rows = table.find_all('tr')
        
        if not rows: continue
        
        # Lấy tiêu đề cột
        headers = []
        header_row = rows[0].find_all(['th', 'td'])
        headers = [clean_wiki_text(h.get_text()) for h in header_row]

        # Lấy dữ liệu từng hàng
        for row in rows[1:]:
            cells = row.find_all(['td', 'th'])
            # Đảm bảo lấy đúng nội dung plain text trong từng ô
            row_data = [clean_wiki_text(c.get_text()) for c in cells]
            
            # Nếu số cột khớp với header, tạo dictionary
            if len(row_data) == len(headers):
                table_row = dict(zip(headers, row_data))
                table_info["data"].append(table_row)
            else:
                # Trường hợp đặc biệt (ô bị gộp - rowspan/colspan), lưu dạng list thô
                table_info["data"].append({"raw_row": row_data})
        
        result_data["tables"].append(table_info)

    # Gộp các mảng text trong sections thành chuỗi duy nhất
    result_data["sections"] = {k: " ".join(v) for k, v in result_data["sections"].items()}

    # Lưu ra file JSON
    with open(output_json, 'w', encoding='utf-8') as out_f:
        json.dump(result_data, out_f, ensure_ascii=False, indent=4)
    
    print(f"Hoàn tất! Đã trích xuất {len(result_data['tables'])} bảng và {len(result_data['sections'])} mục nội dung.")

# Thực thi
extract_comprehensive_data(r'D:\MScThi\12_02_2026\first_trial.txt', 'thủ_khoa_nho_hoc_full.json')

Hoàn tất! Đã trích xuất 2 bảng và 5 mục nội dung.


**SEARCH WIKI EACH CHARACTER**

In [3]:
import json
import os
import requests
import time
from urllib.parse import quote

def download_wiki_characters(json_file, output_folder="thu_khoa_html"):
    # 1. Tạo thư mục lưu trữ nếu chưa có
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # 2. Đọc dữ liệu từ file JSON của bạn
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Danh sách để theo dõi các tên đã lấy (tránh trùng lặp)
    character_names = []

    # Duyệt qua các bảng trong JSON để tìm raw_row
    for table in data.get("tables", []):
        for row in table.get("data", []):
            if "raw_row" in row:
                # Lấy tên ở dòng thứ 2 (index 1)
                name = row["raw_row"][1].strip()
                if name and name not in character_names:
                    character_names.append(name)

    print(f"Tìm thấy {len(character_names)} nhân vật. Bắt đầu tải...")

    # 3. Truy cập Wiki và tải HTML từng người
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    for name in character_names:
        # Tạo URL search trực tiếp hoặc link bài viết
        # Wikipedia Tiếng Việt: https://vi.wikipedia.org/wiki/Tên_Nhân_Vật
        safe_name = quote(name.replace(" ", "_"))
        url = f"https://vi.wikipedia.org/wiki/{safe_name}"
        
        file_path = os.path.join(output_folder, f"{name.replace(' ', '_')}.txt")

        # Kiểm tra nếu file đã tồn tại thì bỏ qua
        if os.path.exists(file_path):
            continue

        try:
            response = requests.get(url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                with open(file_path, "w", encoding="utf-8") as html_file:
                    html_file.write(response.text)
                print(f"√ Đã tải: {name}")
            else:
                print(f"x Không tìm thấy bài viết cho: {name} (Status: {response.status_code})")
            
            # Nghỉ một chút để tránh bị Wiki chặn (Rate limit)
            time.sleep(1) 

        except Exception as e:
            print(f"! Lỗi khi tải {name}: {e}")

    print(f"\n--- HOÀN TẤT ---")
    print(f"Dữ liệu được lưu tại thư mục: {os.path.abspath(output_folder)}")

# Chạy chương trình
if __name__ == "__main__":
    download_wiki_characters('thủ_khoa_nho_hoc_full.json')

Tìm thấy 179 nhân vật. Bắt đầu tải...
x Không tìm thấy bài viết cho: Thủ khoa Thái học sinh, Trại trạng nguyên, Hội nguyên (Status: 404)
√ Đã tải: Lê Văn Thịnh
√ Đã tải: Mạc Hiển Tích
√ Đã tải: Bùi Quốc Khái
√ Đã tải: Trương Hanh
√ Đã tải: Lưu Miễn
√ Đã tải: Nguyễn Quan Quang
√ Đã tải: Nguyễn Hiền
√ Đã tải: Trần Quốc Lặc
√ Đã tải: Trần Cố
√ Đã tải: Lý Đạo Tái
√ Đã tải: Đào Tiêu
√ Đã tải: Mạc Đĩnh Chi
√ Đã tải: Đào Sư Tích
√ Đã tải: Đoàn Xuân Lôi
x Không tìm thấy bài viết cho: Hoàng Quán Chi (Status: 404)
√ Đã tải: Lưu Thúc Kiệm
x Không tìm thấy bài viết cho: Hà Ngạn Thần (Status: 404)
√ Đã tải: Triệu Thái
√ Đã tải: Nguyễn Thiên Tích
x Không tìm thấy bài viết cho: Nguyễn Vết Tuyên (Status: 404)
√ Đã tải: Nguyễn Trực
√ Đã tải: Nguyễn Nghiêu Tư
x Không tìm thấy bài viết cho: Vũ Bá Triệt (Status: 404)
√ Đã tải: Lương Thế Vinh
x Không tìm thấy bài viết cho: Dương Như Châu (Status: 404)
x Không tìm thấy bài viết cho: Phạm Bá (Status: 404)
√ Đã tải: Vũ Kiệt
√ Đã tải: Vũ Tuấn Chiêu
√ Đã tải: L

**EXTRACT EACH CHARACTER**

In [5]:
import os
import json
import re
from bs4 import BeautifulSoup

class BulkWikiParser:
    def __init__(self, input_folder="thu_khoa_html_storage", output_file="database_tong_hop.json"):
        self.input_folder = input_folder
        self.output_file = output_file
        self.processed_titles = set() # Bộ nhớ để khử trùng

    def clean_text(self, text):
        """Làm sạch văn bản: xóa chú thích [1], ký tự lạ và khoảng trắng thừa."""
        if not text: return ""
        # Xóa tham chiếu chú thích dạng [1], [2], [ghi chú...]
        text = re.sub(r'\[[^\]]*\]', '', text)
        # Chuẩn hóa khoảng trắng và ngắt dòng
        text = text.replace('\xa0', ' ').replace('\n', ' ')
        return re.sub(r'\s+', ' ', text).strip()

    def parse_file(self, file_path):
        """Trích xuất chi tiết nội dung từ một file HTML Wikipedia."""
        with open(file_path, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')

        # 1. Tên nhân vật (Tiêu đề bài viết)
        name_tag = soup.find('h1', id='firstHeading')
        name = name_tag.get_text(strip=True) if name_tag else "Unknown"

        # Kiểm tra trùng lặp dựa trên tên bài viết (đã chuẩn hóa)
        normalized_title = name.lower().strip()
        if normalized_title in self.processed_titles or name == "Unknown":
            return None
        self.processed_titles.add(normalized_title)

        # 2. Trích xuất Infobox
        infobox = {}
        info_table = soup.find('table', class_='infobox')
        if info_table:
            for row in info_table.find_all('tr'):
                label = row.find(['th', 'td'], class_='infobox-label') or row.find('th')
                value = row.find(['td'], class_='infobox-data') or row.find('td')
                if label and value and label != value:
                    key = self.clean_text(label.get_text())
                    val = self.clean_text(value.get_text())
                    if key: infobox[key] = val

        # 3. Trích xuất nội dung văn bản theo Sections
        content_body = soup.find('div', class_='mw-parser-output')
        sections = {}
        if content_body:
            # Lấy tóm tắt mở đầu
            intro = []
            for elem in content_body.find_all(['p', 'ul'], recursive=False):
                txt = self.clean_text(elem.get_text())
                if txt: intro.append(txt)
            sections["Giới thiệu"] = " ".join(intro)

            # Duyệt qua các tiêu đề để lấy nội dung chi tiết
            current_header = None
            for elem in content_body.find_all(['h2', 'h3', 'p', 'ul']):
                if elem.name in ['h2', 'h3']:
                    header_text = self.clean_text(elem.get_text().replace('sửa', '').replace('mã nguồn', ''))
                    # Loại bỏ các mục không mang giá trị thông tin nhân vật
                    if header_text in ["Xem thêm", "Sách tham khảo chính", "Chú thích", "Liên kết ngoài", "Nguồn", "Tham khảo"]:
                        current_header = None
                    else:
                        current_header = header_text
                elif current_header and elem.name in ['p', 'ul']:
                    txt = self.clean_text(elem.get_text())
                    if txt:
                        sections.setdefault(current_header, []).append(txt)

            # Gộp list thành string cho mỗi section
            for k in sections:
                if isinstance(sections[k], list):
                    sections[k] = " ".join(sections[k])

        # 4. Trích xuất Thể loại (Categories)
        categories = []
        cat_links = soup.find('div', id='mw-normal-catlinks')
        if cat_links:
            categories = [self.clean_text(li.get_text()) for li in cat_links.find_all('li')]

        return {
            "nhan_vat": name,
            "infobox": infobox,
            "noi_dung_chi_tiet": sections,
            "the_loai": categories
        }

    def run(self):
        """Quét folder và tạo file JSON tổng hợp."""
        all_data = []
        if not os.path.exists(self.input_folder):
            print(f"Lỗi: Thư mục '{self.input_folder}' không tồn tại.")
            return

        files = [f for f in os.listdir(self.input_folder) if f.endswith('.txt')]
        print(f"Bắt đầu xử lý {len(files)} file...")

        for idx, filename in enumerate(files):
            file_path = os.path.join(self.input_folder, filename)
            try:
                data = self.parse_file(file_path)
                if data:
                    all_data.append(data)
                    print(f"[{idx+1}/{len(files)}] √ Xử lý xong: {data['nhan_vat']}")
                else:
                    print(f"[{idx+1}/{len(files)}] - Bỏ qua (Trùng lặp): {filename}")
            except Exception as e:
                print(f"[{idx+1}/{len(files)}] x Lỗi tại {filename}: {e}")

        # Ghi dữ liệu ra file JSON
        with open(self.output_file, 'w', encoding='utf-8') as out_f:
            json.dump(all_data, out_f, ensure_ascii=False, indent=4)
        
        print(f"\n--- HOÀN THÀNH ---")
        print(f"Tổng số nhân vật duy nhất: {len(all_data)}")
        print(f"Kết quả lưu tại: {os.path.abspath(self.output_file)}")

if __name__ == "__main__":
    # Bạn hãy thay đổi tên folder đầu vào cho đúng với máy của mình
    parser = BulkWikiParser(input_folder="thu_khoa_html", output_file="database_thu_khoa_final.json")
    parser.run()

Bắt đầu xử lý 118 file...
[1/118] √ Xử lý xong: Bùi Dương Lịch
[2/118] √ Xử lý xong: Bùi Huy Bích
[3/118] √ Xử lý xong: Bùi Quốc Khái
[4/118] √ Xử lý xong: Bùi Sĩ Tiêm
[5/118] √ Xử lý xong: Dương Phúc Tư
[6/118] √ Xử lý xong: Giang Văn Minh
[7/118] √ Xử lý xong: Giáp Hải
[8/118] √ Xử lý xong: Hoàng Bính
[9/118] √ Xử lý xong: Hoàng Nghĩa Phú
[10/118] √ Xử lý xong: Hoàng Tế Mỹ
[11/118] √ Xử lý xong: Hoàng Văn Tán
[12/118] √ Xử lý xong: Hoàng Đình Tá
[13/118] √ Xử lý xong: Hồ Sĩ Đống
[14/118] √ Xử lý xong: Lê Nại
[15/118] √ Xử lý xong: Lê Quý Đôn
[16/118] √ Xử lý xong: Lê Quảng Chí
[17/118] √ Xử lý xong: Lê Trạc Tú
[18/118] √ Xử lý xong: Lê Văn Thịnh
[19/118] √ Xử lý xong: Lê Ích Mộc
[20/118] √ Xử lý xong: Huyền Quang
[21/118] √ Xử lý xong: Lưu Danh Công
[22/118] √ Xử lý xong: Lưu Miễn (định hướng)
[23/118] √ Xử lý xong: Lưu Thúc Kiệm
[24/118] √ Xử lý xong: Lưu Đình Chất
[25/118] √ Xử lý xong: Lương Thế Vinh
[26/118] √ Xử lý xong: Mai Anh Tuấn
[27/118] √ Xử lý xong: Mạc Hiển Tích
[28/118]